# GatorTron Training and Prediction

## 5-Fold Cross Validation Strategy (Updated 12/23)

### Import & Load Packages

### Note Filtering

In [ ]:
import re
import medspacy
from medspacy.section_detection import Sectionizer
from negspacy.negation import Negex
from spacy.pipeline import EntityRuler
from negspacy.termsets import termset

#load medspacy
nlp=medspacy.load(medspacy_enable='default')

#sentence splitting
nlp.add_pipe("sentencizer") #add sentencizer to pipeline. splits text into individual sentences

#filter by section based on section names in the .json
#adapted from kiersten's code
sectionizer = Sectionizer(nlp, rules="/home/sbalaj4/.config/sections.json",name="custom_sectionizer")

#keywords for sentence filtering - used chatgpt help in creating this function
def load_metastasis_words(filepath):
    keywords = [] #initialize empty list to store extracted words 
    with open(filepath, "r", encoding="utf-8") as f: # filepath: file path to keywords txt file, r: read mode, utf-8: ensures characters load consistently 
        for line in f: #loops line by line through keyword file
            parts = [w.strip().lower() for w in line.split(",") if w.strip()] #comma separate lines, remove trailing white space, lowercase every word for consistency
            keywords.extend(parts) 
    return keywords

filepath = "/home/sbalaj4/expanded_metastasis_terms copy.txt" #specify path to keywords
mets_words = load_metastasis_words(filepath) #save all terms used in the keywords text file

#negation detection
ts=termset("en_clinical") #loads predefined negation patterns from negspacy

#specify negation patterns
patterns=ts.get_patterns() #get all negation phrases used in the termset

#add to nlp pipeline
#add negex detection to pipeline. apply negation detection only to entities labeled as "METS" (metastasis keyword)
nlp.add_pipe("negex", config={"neg_termset":patterns,"ent_types":["METS"],"extension_name":"negex","chunk_prefix":["no"]},last=True) 
ruler = nlp.add_pipe("entity_ruler", before="negex") #add entity for metastasis keyword identification

patterns = [] #hold all entity patterns passed through EntityRule
for w in mets_words: #loop over each word/phrase in the list of metastasis keywords
    parts = w.split(",") #split by comma
    token_pattern=[{"LOWER":p} for p in parts]
    patterns.append({"label":"METS","pattern":token_pattern})

ruler.add_patterns(patterns)

def clean_text(text): 
    text = text.lower() #lower case all text
    return text.strip() #removes trailing white spaces

#identify positive entities in text - got chatgpt help for this
def positive_sent(text, keywords, nlp):
    doc = nlp(text) #passes text through nlp pipeline
    found_positive = False #flags positive/non-negated mentions of keyword
    for ent in doc.ents: #loop through all entities detected in the text
        if ent.label_ == "METS": #checks whether entity is labeled as "METS"
            if not getattr(ent._, "negex", False):  #checks that the entity is not negated, using negex detection
                found_positive = True #if non-negated mention of keyword is found, labeled as "True"
    return found_positive

#preprocess - adapted from Kiersten's code
def preprocess_note(text, nlp, sectionizer, mets_words): #input clinical text, along with nlp pipeline, sectionizer, and keyword file 
    doc = nlp(text) #processes clinical note text
    sectionizer(doc) #sectionizer
    revised_note = [] #initializes "revised_note" to store pre-processed note
    contains_keyword = any(positive_sent(sent.text, mets_words, nlp) for sent in doc.sents) #checks for keywords in each sentence. returns true if non-negated
    if contains_keyword:
        for sent in doc.sents:
            if positive_sent(sent.text, mets_words, nlp):
                revised_note.append(clean_text(sent.text)) #again, loops through sentences to identify keywords and only keeps sentences with non-negative mentions of the keyword.
    else: #if not keywords are found
        for i, title in enumerate(doc._.section_titles): #iterate through each section of the note (sections identified through section titles)
            categories = doc._.section_categories[i] #categories from .json
            body_text = str(doc._.section_bodies[i]) #text 
            if categories is None: #if there are no categories, convert to empty list
                categories = []
            elif isinstance(categories, str):
                categories = [categories]
            categories = [c.lower().strip() for c in categories if c]
            if any("_retain" in c for c in categories) and body_text.strip(): #check if the category contains "_retain"
                revised_note.append(f"{title}\n\t{clean_text(body_text)}") #append cleaned text to "revised_note"

    return "\n".join([s for s in revised_note if s])


### Apply Filtering to Training Data

In [ ]:
input_text=pd.read_csv("/labs/bozkurtlab/metastasis-data/Radiology_SB_NOTES_22pts_061725.csv") #raw notes

#specify column with the notes
filtered_set = input_text[input_text['Full_text'].notna()].copy() # may not necessarily be the column name, but specify the correctname when running this
filtered_set["Full_text"] = filtered_set["Full_text"].astype(str)

#detect mets keywords
filtered_set["contains_keyword"] = filtered_set["Full_text"].apply(
    lambda x: positive_sent(x, mets_words, nlp)
)

filtered_set["revised_note"] = filtered_set.apply(
    lambda row: preprocess_note(row["Full_text"], nlp, sectionizer, mets_words),
    axis=1
)

#save notes
filtered_set.to_csv(["insert path"], index=False)

#### After this, we manually annotated the revised note

### 5-fold CV

In [ ]:
#Import Packages
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,Trainer, TrainingArguments, DataCollatorWithPadding)
from sklearn.model_selection import KFold #for 5-fold CV
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

#Define locally saved GatorTron path and data 
model_id="/home/sbalaj4/gatortron-base" #locally saved GatorTron
data_path="/home/sbalaj4/Sweta-annotations-Preprocessed_Radiology_Notes_22pts.csv" #My annotated notes (which already contain the filtered note)

#Load Data
df = pd.read_csv(data_path) #read in data file
df["revised_note"] = df["revised_note"].astype(str) #convert filtered note column to string
df["metastasis"] = df["metastasis"].astype(int) #convert mets labels to integers
df["cns"] = df["cns"].astype(int) #convert cns mets labels to integers

tokenizer = AutoTokenizer.from_pretrained(model_id)

#Note Chunking
def chunk_notes(text_list): #take revised note text and turn them into chunks
    ids, masks, note_ids, chunk_ids = [], [], [], [] #initialize: id = token id, masks, note_id = which note the chunk came from, chunk_ids = which number chunk inside the note
    for note_i, text in enumerate(text_list): #helps identify which note thechunk came from
        enc = tokenizer(text, truncation=True, return_overflowing_tokens=True,max_length=512,stride=128,return_attention_mask=True)
        #this above tokenizes the fill note, splits into chunks based on length of 512, adds overlap between chunk (to help with context clarity)

        for c, (inp, att) in enumerate(zip(enc["input_ids"], enc["attention_mask"])):
            ids.append(inp)
            masks.append(att)
            note_ids.append(note_i)
            chunk_ids.append(c)
        #This above loops over chunks from each note. This enc["input_ids"] is a list of chunks.
        #We loop over each chunk c = chunk index, inp = token IDs for that chunk, att = attention mask for that chunk

    return {
        "input_ids": ids,
        "attention_mask": masks,
        "note_id": note_ids,
        "chunk_id": chunk_ids
    }

#Got some help from ChatGPT on this part
class NotesDataset(Dataset): #Take chunked notes and have pytorch iterate over it during training and eval
    def __init__(self, encodings, labels=None): #store encodings (input id, attention mask, note id, chunk id for each chunk)
        self.encodings = encodings
        self.labels = labels
    
    def __len__(self):
        return len(self.encodings["input_ids"]) #def number of training examples, based on input ids
    
    def __getitem__(self, idx): #define what a training example looks like
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items() if key != "labels"} # build one training example. for each idx, you get what is stored in the encoding.
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx]) 
        return item

def make_dataset(df, label_col): #convert note-level dataset to a chunk-level one so that chunks can get labels that are then mapped back to the notes.
    encodings = chunk_notes(df["revised_note"].tolist()) #apply chunk notes function
    labels = [df[label_col].iloc[n] for n in encodings["note_id"]] 
    return NotesDataset(encodings, labels), encodings

#Note-level Prediction
def note_level_predictions(dataset, trainer, encodings): #one prediction produced per note after running inferences on chunks
    trainer.model.eval()
    preds = trainer.predict(dataset) #chunk level predictions
    logits = preds.predictions
    probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy() #turns logits into probabilities
    note_probs = {}
    for prob, note_id in zip(probs, encodings["note_id"]): #group chunk probabilities by note. each note contains all of its chunk probabilities for mets/cns labels
        note_probs.setdefault(note_id, []).append(prob)
    note_preds = {nid: 1 if max(probs) >= 0.5 else 0 for nid, probs in note_probs.items()} #If any chunk has a probability >0.5 for mets presence or cns presence, than the entire note gets a 1. This is an arbitrary threshold though.
    return note_preds #return 0s or 1s depending on the probabilities

# 5-fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42) #set 5 folds. sets a seed

for target in ["metastasis", "cns"]: #training separate sequence classifiers: one for mets presence and another for cns mets
    if target == "cns": #Only train cns classifier when mets presence = 1
        df_target = df[df["metastasis"] == 1].reset_index(drop=True)
    else:
        df_target = df.copy()
        
    for fold, (train_idx, val_idx) in enumerate(kf.split(df_target)): # each time, use 4 folds for training and 1 for validation
        train_df = df_target.iloc[train_idx].reset_index(drop=True) #split training (at the note level)
        val_df = df_target.iloc[val_idx].reset_index(drop=True) #split validation (at the note level)
        
        train_ds, train_enc = make_dataset(train_df, target)
        val_ds, val_enc = make_dataset(val_df, target)
        
        model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
        
        #Training setup
        output_dir = f"./{target}_fold{fold+1}" #target is either metastasis or cns
        training_args = TrainingArguments(
            output_dir=output_dir,
            per_device_train_batch_size=4,
            per_device_eval_batch_size=4,
            learning_rate=2e-5,
            num_train_epochs=5,
            logging_dir=os.path.join(output_dir, "logs"),
            logging_steps=200,
            save_strategy="no",
        )
        
        data_collator = DataCollatorWithPadding(tokenizer)
        
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            tokenizer=tokenizer,
            data_collator=data_collator,
        )
        
        trainer.train()
        
        #Evaluate note level
        note_preds = note_level_predictions(val_ds, trainer, val_enc)
        note_labels = val_df[target].to_dict()

        #Get accuracy, precision, recall, f1 (Got some ChatGPT help on this section)
        y_true = [note_labels[nid] for nid in note_preds.keys()]
        y_pred = [note_preds[nid] for nid in note_preds.keys()]

        p, r, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average="binary"
        )
        acc = accuracy_score(y_true, y_pred)

        print(f"Fold {fold+1} note-level metrics:")
        print({
            "accuracy": acc,
            "precision": p,
            "recall": r,
            "f1": f1
        })



### Validation on new, unseen notes

Ensembling all the folds for metastasis presence and CNS, respectively

### Apply Filtering to Unseen Notes

In [ ]:
#adapted from Kiersten's code
input_text=pd.read_csv("/labs/bozkurtlab/metastasis-data/Preprocessed_ALL_Radiology_Notes.csv")

#specify column with the notes - 'TEXT' column (or the name of the column containing the full text)
filtered_set = input_text[input_text['TEXT'].notna()].copy()
filtered_set["TEXT"] = filtered_set["TEXT"].astype(str)

#detect mets keywords
filtered_set["contains_keyword"] = filtered_set["TEXT"].apply(
    lambda x: positive_sent(x, mets_words, nlp)
)

filtered_set["revised_note"] = filtered_set.apply(
    lambda row: preprocess_note(row["TEXT"], nlp, sectionizer, mets_words),
    axis=1
)

#remove any notes that are empty after filtering
filtered_set = filtered_set[filtered_set['revised_note'].str.strip() != '']
print(f"Total notes after preprocessing: {len(filtered_set)}")

#save notes
filtered_set.to_csv("/labs/bozkurtlab/metastasis-data/Preprocessed_ALL_Radiology_Notes_FILT.csv", index=False)

### Apply Fine-tuned Model: Metastasis presence

In [ ]:
target="metastasis"
num_folds=5
data_path=("/labs/bozkurtlab/metastasis-data/Preprocessed_ALL_Radiology_Notes_FILT.csv")
max_len=512
stride=128
batch_size=4

#New data
df_new=pd.read_csv(data_path)
df_new["revised_note"]=df_new["revised_note"].astype(str) #convert revised note column to string

#Get tokenizer from the first fold, since all folds have the same one
tokenizer=AutoTokenizer.from_pretrained(f"./{target}_fold1")

#Note Chunking
def chunk_notes(text_list): #take revised note text and turn them into chunks
    ids, masks, note_ids, chunk_ids = [], [], [], [] #initialize: id = token id, masks, note_id = which note the chunk came from, chunk_ids = which number chunk inside the note
    for note_i, text in enumerate(text_list): #helps identify which note thechunk came from
        enc = tokenizer(text, truncation=True, return_overflowing_tokens=True,max_length=512,stride=128,return_attention_mask=True)
        #this above tokenizes the fill note, splits into chunks based on length of 512, adds overlap between chunk (to help with context clarity)

        for c, (inp, att) in enumerate(zip(enc["input_ids"], enc["attention_mask"])):
            ids.append(inp)
            masks.append(att)
            note_ids.append(note_i)
            chunk_ids.append(c)
        #This above loops over chunks from each note. This enc["input_ids"] is a list of chunks.
        #We loop over each chunk c = chunk index, inp = token IDs for that chunk, att = attention mask for that chunk

    return {
        "input_ids": ids,
        "attention_mask": masks,
        "note_id": note_ids,
        "chunk_id": chunk_ids
    }

class NotesDataset(torch.utils.data.Dataset):
    def __init__(self,encodings):
        self.encodings=encodings
    def __len__(self):
        return len(self.encodings["input_ids"])
    def __getitem__(self,idx):
        return {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}

encodings=chunk_notes(df_new["revised_note"].tolist()) #apply chunking to new notes
dataset=NotesDataset(encodings)
data_collator=DataCollatorWithPadding(tokenizer)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
for m in models:
    m.to(device)

model=[] #load all fold models
for fold in range(1,num_folds+1):
    model_dir=f"./{target}_fold{fold}"
    model=AutoModelForSequenceClassification.from_pretrained(model_dir)
    model.eval()
    models.append(model)

#Ensemble - Got ChatGPT help for this part
probs=[[]for _ in range(len(encodings["input_ids"]))]

for model in models:
    loader=torch.utils.data.DataLoader(dataset,batch_size=batch_size,collate_fn=data_collator)
    for batch_idx,batch in enumerate(loader):
        batch={k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            logits=model(**batch).logitsprobs=torch.softmax(logits,dim=1)[:,1]
        for i, p in enumerate(probs.cpu().numpy()):
            all_probs[batch_idx*batch_size+i].append(p)

avg_probs=[np.mean(p_list) for p_list in probs]

#note level predictions
note_probs={}
for prob, note_id in zip(avg_probs,encodings["note_id"]):
    note_probs.setdefault(note_id,[]).append(prob)

note_preds={nid:1 if max(probs) >=0.5 else 0 for nid, probs in note_probs.items()}

#Predictions to new column
df_new["prediction_mets"]=df_new.index.map(lambda idx: not_preds.get(idx,0))

### Apply Fine-tuned Model: CNS Metastasis presence

In [ ]:
target="cns"
num_folds=5
data_path=("/labs/bozkurtlab/metastasis-data/Preprocessed_ALL_Radiology_Notes_FILT.csv")
max_len=512
stride=128
batch_size=4

#New data
df_new=pd.read_csv(data_path)
df_new["revised_note"]=df_new["revised_note"].astype(str) #convert revised note column to string

#Get tokenizer from the first fold, since all folds have the same one
tokenizer=AutoTokenizer.from_pretrained(f"./{target}_fold1")

#Note Chunking
def chunk_notes(text_list): #take revised note text and turn them into chunks
    ids, masks, note_ids, chunk_ids = [], [], [], [] #initialize: id = token id, masks, note_id = which note the chunk came from, chunk_ids = which number chunk inside the note
    for note_i, text in enumerate(text_list): #helps identify which note thechunk came from
        enc = tokenizer(text, truncation=True, return_overflowing_tokens=True,max_length=512,stride=128,return_attention_mask=True)
        #this above tokenizes the fill note, splits into chunks based on length of 512, adds overlap between chunk (to help with context clarity)

        for c, (inp, att) in enumerate(zip(enc["input_ids"], enc["attention_mask"])):
            ids.append(inp)
            masks.append(att)
            note_ids.append(note_i)
            chunk_ids.append(c)
        #This above loops over chunks from each note. This enc["input_ids"] is a list of chunks.
        #We loop over each chunk c = chunk index, inp = token IDs for that chunk, att = attention mask for that chunk

    return {
        "input_ids": ids,
        "attention_mask": masks,
        "note_id": note_ids,
        "chunk_id": chunk_ids
    }

class NotesDataset(torch.utils.data.Dataset):
    def __init__(self,encodings):
        self.encodings=encodings
    def __len__(self):
        return len(self.encodings["input_ids"])
    def __getitem__(self,idx):
        return {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}

encodings=chunk_notes(df_new["revised_note"].tolist()) #apply chunking to new notes
dataset=NotesDataset(encodings)
data_collator=DataCollatorWithPadding(tokenizer)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
for m in models:
    m.to(device)

model=[] #load all fold models
for fold in range(1,num_folds+1):
    model_dir=f"./{target}_fold{fold}"
    model=AutoModelForSequenceClassification.from_pretrained(model_dir)
    model.eval()
    models.append(model)

#Ensemble - Got ChatGPT help for this nextpart
probs=[[]for _ in range(len(encodings["input_ids"]))]

for model in models:
    loader=torch.utils.data.DataLoader(dataset,batch_size=batch_size,collate_fn=data_collator)
    for batch_idx,batch in enumerate(loader):
        batch={k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            logits=model(**batch).logitsprobs=torch.softmax(logits,dim=1)[:,1]
        for i, p in enumerate(probs.cpu().numpy()):
            all_probs[batch_idx*batch_size+i].append(p)

avg_probs=[np.mean(p_list) for p_list in probs]

#note level predictions
note_probs={}
for prob, note_id in zip(avg_probs,encodings["note_id"]):
    note_probs.setdefault(note_id,[]).append(prob)

note_preds={nid:1 if max(probs) >=0.5 else 0 for nid, probs in note_probs.items()}

#Predictions to new column
df_new["prediction_cns"]=df_new.index.map(lambda idx: not_preds.get(idx,0))

### After this, we would annotate the new, unseen notes to see how well the predictions match up